In [ ]:
import tensorflow as tf
import keras
import glob
import os

Check current runtime

In [2]:
print("Tensorflow version: ",tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU')) # check for GPU

Tensorflow version:  2.21.0
Available GPUs: []


Download & un the data

In [3]:
def get_data_extract():
  if "dataset" in os.listdir():
    print("Dataset already exists")
  else:
    print("Downloading the data...")
    !wget -O food-data.zip https://www.kaggle.com/api/v1/datasets/download/trolukovich/food11-image-dataset
    print("Dataset downloaded!")
    print("Extracting data..")
    !mkdir dataset
    !unzip -q food-data.zip -d dataset
    print("Extraction done!")

get_data_extract()

Dataset downloaded!
Extracting data..


'wget' is not recognized as an internal or external command,
operable program or batch file.


Extraction done!


'unzip' is not recognized as an internal or external command,
operable program or batch file.


# Dataset

Create Dataset from list of path

In [4]:
path = glob.glob("dataset/*/*/*.jpg")
label = [i.split(".")[0].split("/")[-2] for i in path]

Image augmentation

In [5]:
image_width, image_height = 224,244 # this may need to be changed based on loaded model

aug = keras.Sequential([
    keras.layers.Resizing(image_width, image_height),
    keras.layers.Rescaling(1./255), # this may need to be changed based on loaded model
])

Dataloader

In [6]:
def load_image(path, label, aug):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    # image augmentation
    image = aug(image)
    return image, label

In [13]:
# Corrected Dataset preparation using map
ds = tf.data.Dataset.from_tensor_slices((path, label))
ds = ds.map(lambda path, label: load_image(path, label, aug))  # Ensure load_image() gets path, label, and augmentation
ds = ds.shuffle(buffer_size=1000)  # Shuffle dataset after map
ds = ds.batch(16).prefetch(tf.data.AUTOTUNE)

TypeError: in user code:

    File "C:\Users\hosam\AppData\Local\Temp\ipykernel_15864\4174389986.py", line 3, in None  *
        lambda path, label: load_image(path, label, aug)
    File "C:\Users\hosam\AppData\Local\Temp\ipykernel_15864\1721429258.py", line 2, in load_image  *
        image = tf.io.read_file(path)

    TypeError: Input 'filename' of 'ReadFile' Op has type float32 that does not match expected type of string.


Display Sample from our custom dataset

In [54]:
print("Sample from the dataset:",next(iter(ds))[1])

Sample from the dataset: tf.Tensor(
[b'Soup' b'Egg' b'Dessert' b'Dessert' b'Dairy product' b'Noodles-Pasta'
 b'Soup' b'Bread'], shape=(8,), dtype=string)


# Model